# Telugu Gita RAG — experiment bench

All embedding and evaluation on GPU. The same work takes ~1 hour per run on a
4-core CPU and ~1-2 minutes here.

**Runtime → Change runtime type → T4 GPU** first.

Needs on Drive (`MyDrive/telugu-rag/`): `bge-m3-gita-final`, `bge-reranker-gita-final`.
Upload `gita_experiments_data.zip` when prompted.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
%pip install -q -U sentence-transformers rank_bm25


## Setup


In [ ]:
from google.colab import drive, files
import zipfile, os, json
drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive/telugu-rag'
FT   = f'{DRIVE}/bge-m3-gita-final'
RER  = f'{DRIVE}/bge-reranker-gita-final'
for p in (FT, RER): print(p, '->', 'OK' if os.path.exists(p) else 'MISSING')


In [ ]:
up = files.upload()          # gita_experiments_data.zip
for n in up:
    if n.endswith('.zip'): zipfile.ZipFile(n).extractall('.')
def jl(p): return [json.loads(l) for l in open(p,encoding='utf-8') if l.strip()]
corpus = jl('gita.jsonl')
enrich = {r['verse_id']: r for r in jl('enrichment.jsonl')}
held   = jl('held_queries.jsonl')
print(f'corpus={len(corpus)}  enriched={len(enrich)}  held={len(held)}')


## Shared pieces

`enrichment_text` and `bm25_tokens` must match `app/retrieval.py` exactly, or
the numbers here will not describe the local system.


In [ ]:
import re, math, unicodedata, numpy as np
from collections import defaultdict

def norm(s): return unicodedata.normalize('NFC', s or '').strip()

def retrieval_text(r):
    parts=[r.get('tatparyam_te',''), r.get('adhyaya','')]
    return norm(' '.join(p for p in parts if p))

def enrichment_text(r):
    parts=(r.get('themes_te') or [])+(r.get('life_situations_te') or [])+(r.get('emotions_te') or [])
    return norm(' . '.join(p for p in parts if p))

def bm25_tokens(s):
    s=norm(s); return [s[i:i+4] for i in range(max(0,len(s)-3))] + [w for w in s.split() if w]

def rrf(lists, k=60):
    out=defaultdict(float)
    for lst in lists:
        for i,d in enumerate(lst): out[d]+=1.0/(k+i+1)
    return out

def ndcg_at(rank,k=10): return 1.0/math.log2(rank+1) if (rank and rank<=k) else 0.0

ids=[r['id'] for r in corpus]
by_id={r['id']:r for r in corpus}
tat_texts=[retrieval_text(r) for r in corpus]
enr_ids=[i for i in ids if i in enrich and enrichment_text(enrich[i])]
enr_texts=[enrichment_text(enrich[i]) for i in enr_ids]
print(len(tat_texts),'tatparyam |',len(enr_texts),'enrichment')


## Embed everything once, on GPU

Four matrices: {fine-tuned, base} x {tatparyam, enrichment}. Every experiment
below reuses these, so nothing is embedded twice.


In [ ]:
from sentence_transformers import SentenceTransformer
import torch, gc

EMB={}
for tag, path in [('ft', FT), ('base', 'BAAI/bge-m3')]:
    m=SentenceTransformer(path, device='cuda'); m.max_seq_length=512
    enc=lambda xs: np.asarray(m.encode(xs,batch_size=64,normalize_embeddings=True,
                                       show_progress_bar=True))
    EMB[(tag,'tat')]=enc(tat_texts)
    EMB[(tag,'enr')]=enc(enr_texts)
    EMB[(tag,'q')]  =enc([r['query'] for r in held])
    del m; gc.collect(); torch.cuda.empty_cache()
    print(tag,'done')
print({k:v.shape for k,v in EMB.items()})


## Experiment 1 — which model reads enrichment text better?

The fine-tuned model was trained only on (query -> tatparyam) pairs, so it may
be mis-specialised for life-situation text. Ranks the gold verse among all
enrichment vectors, thematic queries only.


In [ ]:
ti={v:k for k,v in enumerate(ids)}; ei={v:k for k,v in enumerate(enr_ids)}
th=[i for i,r in enumerate(held) if r['kind']=='thematic' and r['verse_id'] in ei]
print(f'{len(th)} thematic queries\n')
print(f"{'model':<12}{'surface':<12}{'top1':>7}{'top10':>8}{'top50':>8}{'med':>6}{'mean_sim':>10}")
for tag in ('ft','base'):
    for surf, M, imap in (('tatparyam',EMB[(tag,'tat')],ti), ('enrichment',EMB[(tag,'enr')],ei)):
        ranks=[];sims=[]
        for i in th:
            g=held[i]['verse_id']
            if g not in imap: continue
            s=M@EMB[(tag,'q')][i]; gs=s[imap[g]]
            ranks.append(int((s>gs).sum())+1); sims.append(float(gs))
        r=np.array(ranks)
        print(f'{tag:<12}{surf:<12}{np.mean(r==1):>7.3f}{np.mean(r<=10):>8.3f}'
              f'{np.mean(r<=50):>8.3f}{int(np.median(r)):>6}{np.mean(sims):>10.3f}')


## Experiment 2 — hybrid retrieval, enrichment on/off

Dense (max-pooled over surfaces) + BM25, fused by RRF. No reranker yet.


In [ ]:
from rank_bm25 import BM25Okapi
bm25=BM25Okapi([bm25_tokens(t) for t in tat_texts])

def retrieve(qi, tag='ft', use_enr=True, pool=50):
    q=EMB[(tag,'q')][qi]
    sims={ids[j]: float(EMB[(tag,'tat')][j]@q) for j in range(len(ids))}
    if use_enr:
        es=EMB[(tag,'enr')]@q
        for j,vid in enumerate(enr_ids): sims[vid]=max(sims[vid], float(es[j]))
    dense=[d for d,_ in sorted(sims.items(), key=lambda kv:-kv[1])[:pool]]
    sc=bm25.get_scores(bm25_tokens(held[qi]['query']))
    lex=[ids[j] for j in np.argsort(-sc)[:pool]]
    return [d for d,_ in sorted(rrf([dense,lex]).items(), key=lambda kv:-kv[1])]

def score(rank_fn, tag=''):
    agg=defaultdict(lambda:{'n':0,'h10':0,'h50':0,'nd':0.0}); tot=defaultdict(float); n=0
    for qi,r in enumerate(held):
        ranked=rank_fn(qi); g=r['verse_id']
        rank=ranked.index(g)+1 if g in ranked else None
        nd=ndcg_at(rank); n+=1
        tot['h1']+= 1 if rank==1 else 0
        tot['h10']+= 1 if (rank and rank<=10) else 0
        tot['h50']+= 1 if (rank and rank<=50) else 0
        tot['nd']+=nd
        a=agg[r['kind']]; a['n']+=1; a['nd']+=nd
        if rank and rank<=10: a['h10']+=1
        if rank and rank<=50: a['h50']+=1
    print(f"{tag:<34} r@1={tot['h1']/n:.3f} r@10={tot['h10']/n:.3f} "
          f"r@50={tot['h50']/n:.3f} NDCG={tot['nd']/n:.3f}")
    for k,v in sorted(agg.items()):
        print(f"    {k:<9} n={v['n']:<4} r@10={v['h10']/v['n']:.3f} "
              f"r@50={v['h50']/v['n']:.3f} NDCG={v['nd']/v['n']:.3f}")
    return tot['h10']/n

score(lambda qi: retrieve(qi, 'ft',  use_enr=False), 'ft, no enrichment')
score(lambda qi: retrieve(qi, 'ft',  use_enr=True ), 'ft + enrichment(ft vectors)')
score(lambda qi: retrieve(qi, 'base',use_enr=True ), 'base + enrichment(base vectors)')


## Experiment 3 — two-model routing

Your proposal: fine-tuned model for tatparyam, base model for life situations.
Cross-model scores are not comparable, so the two ranked lists are fused by
RANK (RRF) rather than by score.


In [ ]:
def retrieve_split(qi, pool=50, w_enr=1.0):
    qf=EMB[('ft','q')][qi]; qb=EMB[('base','q')][qi]
    tat=EMB[('ft','tat')]@qf
    dense_t=[ids[j] for j in np.argsort(-tat)[:pool]]
    es=EMB[('base','enr')]@qb
    dense_e=[enr_ids[j] for j in np.argsort(-es)[:pool]]
    sc=bm25.get_scores(bm25_tokens(held[qi]['query']))
    lex=[ids[j] for j in np.argsort(-sc)[:pool]]
    fused=rrf([dense_t,lex])
    for i,d in enumerate(dense_e): fused[d]+=w_enr*(1.0/(60+i+1))
    return [d for d,_ in sorted(fused.items(), key=lambda kv:-kv[1])]

for w in (0.0, 0.5, 1.0, 2.0):
    score(lambda qi,w=w: retrieve_split(qi, w_enr=w), f'split-model, enr weight={w}')


## Experiment 4 — reranker, and pool sweep

One rerank pass over the top 50; smaller pools are read off by truncation, so
pool is the only variable.


In [ ]:
from sentence_transformers import CrossEncoder
ce=CrossEncoder(RER, max_length=192, device='cuda')

def sweep(rank_fn, tag, pools=(10,15,20,30,50), both_surfaces=False):
    cands=[rank_fn(qi)[:50] for qi in range(len(held))]
    flat=[(held[qi]['query'], by_id[c]['tatparyam_te']) for qi,cs in enumerate(cands) for c in cs]
    S=ce.predict(flat, batch_size=256, show_progress_bar=True)
    if both_surfaces:
        flat2=[(held[qi]['query'], enrichment_text(enrich[c]) if c in enrich else '')
               for qi,cs in enumerate(cands) for c in cs]
        S2=ce.predict(flat2, batch_size=256, show_progress_bar=True)
        S=np.maximum(np.asarray(S), np.asarray(S2))
    S=np.asarray(S); off=0; per_q=[]
    for cs in cands:
        per_q.append(S[off:off+len(cs)]); off+=len(cs)
    print(f'\n{tag}')
    print(f"{'pool':>6}{'r@1':>8}{'r@10':>8}{'NDCG':>8}{'ceiling':>9}")
    for pool in pools:
        h1=h10=0; nd=0.0; ceil=0
        for qi,(cs,s) in enumerate(zip(cands,per_q)):
            c=cs[:pool]; g=held[qi]['verse_id']
            if g in c: ceil+=1
            order=[c[j] for j in np.argsort(-s[:pool])]
            rank=order.index(g)+1 if g in order else None
            if rank==1: h1+=1
            if rank and rank<=10: h10+=1; nd+=ndcg_at(rank)
        n=len(held)
        print(f'{pool:>6}{h1/n:>8.3f}{h10/n:>8.3f}{nd/n:>8.3f}{ceil/n:>9.3f}')

sweep(lambda qi: retrieve(qi,'ft',use_enr=False), 'A: no enrichment')
sweep(lambda qi: retrieve(qi,'ft',use_enr=True),  'B: enrichment, rerank tatparyam')
sweep(lambda qi: retrieve(qi,'ft',use_enr=True),  'C: enrichment, rerank BOTH', both_surfaces=True)
sweep(lambda qi: retrieve_split(qi),              'D: split-model routing')
